# Smart Coach Pipeline Setup & Verification

Run each cell in order to verify and fix your setup.

## 1. Check Python Version

In [ ]:
import sys
print(f"Python {sys.version}")
print(f"Executable: {sys.executable}")

version = sys.version_info
if version.major == 3 and version.minor >= 10:
    print("✓ Version compatible")
else:
    print("✗ Need Python 3.10+")

## 2. Check Installed Packages

In [ ]:
packages = {
    'torch': 'torch',
    'torchvision': 'torchvision',
    'cv2': 'opencv-python',
    'numpy': 'numpy',
    'mediapipe': 'mediapipe',
    'ultralytics': 'ultralytics',
    'pandas': 'pandas',
    'scipy': 'scipy'
}

missing = []
for import_name, pkg_name in packages.items():
    try:
        mod = __import__(import_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"✓ {pkg_name:20} {version}")
    except ImportError:
        print(f"✗ {pkg_name:20} MISSING")
        missing.append(pkg_name)

if missing:
    print(f"\n⚠ Missing: {', '.join(missing)}")
    print("Run: !pip install -r requirements.txt")
else:
    print("\n✓ All packages installed")

## 3. Install Missing Packages (if needed)

In [ ]:
# Uncomment and run if packages are missing
# !pip install -r requirements.txt

## 4. Check Model Files

In [ ]:
from pathlib import Path

models_dir = Path("data/models")
required = {
    "yolov8m-pose.pt": "YOLO Pose",
    "yolov8n-face.pt": "YOLO Face",
    "hand_landmarker.task": "MediaPipe Hands"
}

missing_models = []
for filename, desc in required.items():
    filepath = models_dir / filename
    if filepath.exists():
        size = filepath.stat().st_size / (1024*1024)
        print(f"✓ {desc:20} {size:6.1f} MB - {filename}")
    else:
        print(f"✗ {desc:20} MISSING - {filename}")
        missing_models.append(filename)

if missing_models:
    print(f"\n⚠ Missing {len(missing_models)} model(s)")
    print("Need to download - see next cell")
else:
    print("\n✓ All models present")

## 5. Download Missing Models

In [ ]:
# Run this to download models
# Method 1: Use the download script
!python3 scripts/tools/download_models.py

In [ ]:
# Method 2: Manual download (if script fails)
from ultralytics import YOLO
import shutil
from pathlib import Path

print("Downloading YOLO models...")

# Download pose model
pose_model = YOLO('yolov8m-pose.pt')
print("✓ Pose model downloaded to cache")

# Download face model (generic yolov8n)
face_model = YOLO('yolov8n.pt')
print("✓ Face model downloaded to cache")

# Copy to data/models/
cache_dir = Path.home() / ".cache" / "ultralytics"
models_dir = Path("data/models")
models_dir.mkdir(parents=True, exist_ok=True)

# Find and copy
for model_file in ["yolov8m-pose.pt", "yolov8n.pt"]:
    for f in cache_dir.rglob(model_file):
        dest = models_dir / (model_file if "pose" in model_file else "yolov8n-face.pt")
        shutil.copy(f, dest)
        size = dest.stat().st_size / (1024*1024)
        print(f"✓ Copied {dest.name} ({size:.1f} MB)")
        break

print("\n✓ Models ready!")

## 6. Check Test Video

In [ ]:
video_path = Path("data/input/test_video.mp4")

if video_path.exists():
    size = video_path.stat().st_size / (1024*1024)
    print(f"✓ Test video found ({size:.1f} MB)")
    
    # Get video info
    import cv2
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    cap.release()
    
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps:.1f}")
    print(f"  Frames: {frame_count}")
    print(f"  Duration: {duration:.1f}s")
else:
    print("✗ Test video not found")
    print("  Copy your video: cp your_video.mp4 data/input/test_video.mp4")

## 7. Final Verification

In [ ]:
# Run the official verify script
!python3 scripts/tools/verify_setup.py

## 8. Run the Pipeline!

In [ ]:
# This will take a few minutes depending on video length
# Output will be in data/output/
!python3 scripts/processing/run_pipeline.py

## 9. Check Output

In [ ]:
import pandas as pd
from pathlib import Path

output_video = Path("data/output/output_full.mp4")
output_csv = Path("data/output/analytics.csv")

if output_video.exists():
    size = output_video.stat().st_size / (1024*1024)
    print(f"✓ Output video: {output_video} ({size:.1f} MB)")
else:
    print(f"✗ Output video not found")

if output_csv.exists():
    df = pd.read_csv(output_csv)
    print(f"\n✓ Analytics CSV: {output_csv}")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    print(f"\nFirst few columns: {list(df.columns[:10])}")
    print(f"\nFirst few rows:")
    display(df.head())
else:
    print(f"✗ Analytics CSV not found")